In [1]:
# import subprocess
# result = subprocess.run(["which", "tesseract"], capture_output=True, text=True)
# print(result.stdout)  # shows the actual path

# 1. Basic PDF Preprocessing with Unstructured

This extracts structured semantic elements from PDF.

In [2]:
# import os
# os.environ["PATH"] += ":/opt/homebrew/bin/tesseract"  # Apple Silicon M1/M2/M3

import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]

In [3]:
from unstructured.partition.pdf import partition_pdf

/Users/adityabhagwat/Projects/Unstructured-io-Document-Processing-Pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
elements = partition_pdf(
    filename="Training_Data/AML_NOTES_UNIT_1_2_3_4_5_merged.pdf",
    # Best strategy for RAG pipelines
    strategy="hi_res",
    # OCR languages
    languages=["eng"],
    # Keep metadata
    include_metadata=True,
    # Detect tables
    infer_table_structure=True,
    # Extract images/tables if needed
    extract_images_in_pdf=False,
)

In [ ]:
print(f"Total elements: {len(elements)}")

In [ ]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 80)

# 2. Normalize the Output

Unstructured returns heterogeneous element objects:

* Title
* NarrativeText
* ListItem
* Table
* Header
* Footer
* etc.

You should normalize them into ONE stable schema.

In [ ]:
{
  "id": str,
  "document_id": str,
  "chunk_id": str,
  "text": str,
  "element_type": str,
  "page_number": int,
  "section": str,
  "source_file": str,
  "created_at": str,
  "embedding_model": str,
}

In [ ]:
normalized_docs = []

for idx, el in enumerate(elements):

    doc = {
        "id": f"doc_{idx}",
        "type": el.category,
        "text": el.text,
        "page_number": getattr(el.metadata, "page_number", None),
        "filename": getattr(el.metadata, "filename", None),
        "languages": getattr(el.metadata, "languages", None),
        "coordinates": str(getattr(el.metadata, "coordinates", None)),
        "source": getattr(el.metadata, "source", None),
    }
    normalized_docs.append(doc)

In [ ]:
print(normalized_docs[0])

# 3. Chunk the Document Properly

Use Unstructured chunking because it preserves semantic structure.

Why chunk_by_title() is better:

* respects headings
* preserves sections
* prevents chunk mixing across topics
* ideal for PDFs

In [ ]:
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(
    elements,
    max_characters=1200,
    new_after_n_chars=1000,
    # combine tiny sections
    combine_text_under_n_chars=200,
)

In [ ]:
print(f"Total chunks: {len(chunks)}")

In [ ]:
for chunk in chunks[:3]:
    print(chunk.text)
    print("=" * 80)

# 4. Create Embeddings

Using SentenceTransformer model name "sentence-transformers/all-MiniLM-L6-v2" to create embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [chunk.text for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

# Bonus Step

# 5. Insert Embedding into ChromaDB

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="single_pdf_collection"
)

In [ ]:
for i, chunk in enumerate(chunks):
    collection.add(
        ids=[f"chunk_{i}"],
        documents=[chunk.text],
        embeddings=[embeddings[i].tolist()],
        metadatas=[{
            "source": "sample.pdf",
            "chunk_id": i,
        }]
    )

print("Inserted into ChromaDB")

In [ ]:
query = "What is k means clustering?"


In [ ]:
query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print(results["documents"])